
# Data & Feature Extraction Pipeline

This notebook demonstrates how to build a dataset for the **Voice Deepfake Vishing Detection & Response System**.
We use a single real voice recording (`sample.wav`) and create multiple `real` and `fake` samples by slicing and modifying it.
Features such as MFCCs, spectral centroid, bandwidth, rolloff, jitter, and shimmer are computed for each segment and stored in `features.csv`.


In [ ]:

import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal
import scipy.fftpack as fftpack



In [ ]:

# Load the sample audio file
sample_rate, data = wavfile.read('sample.wav')

# If stereo, convert to mono
if data.ndim > 1:
    data = data.mean(axis=1)

# Normalize to [-1, 1]
if data.dtype != np.float32:
    data = data.astype(np.float32)
    data = data / np.max(np.abs(data))

segment_length = sample_rate  # 1 second segments


In [ ]:

def compute_features(segment, sr):
    '''Compute MFCC (13), spectral centroid, bandwidth, rolloff, jitter, and shimmer for a given audio segment.'''
    # Pre-emphasis
    pre_emphasis = 0.97
    emphasized = np.append(segment[0], segment[1:] - pre_emphasis * segment[:-1])

    # Framing
    frame_length = 0.025  # 25ms
    frame_step = 0.010    # 10ms
    frame_length_samples = int(round(frame_length * sr))
    frame_step_samples  = int(round(frame_step * sr))
    signal_length = len(emphasized)
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length_samples)) / frame_step_samples))

    pad_signal_length = num_frames * frame_step_samples + frame_length_samples
    pad_signal = np.append(emphasized, np.zeros(pad_signal_length - signal_length))

    indices = (np.tile(np.arange(0, frame_length_samples), (num_frames, 1)) +
               np.tile(np.arange(0, num_frames * frame_step_samples, frame_step_samples), (frame_length_samples, 1)).T)
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    frames *= np.hamming(frame_length_samples)

    NFFT = 512
    mag_frames = np.absolute(np.fft.rfft(frames, NFFT))
    pow_frames = ((1.0 / NFFT) * (mag_frames ** 2))

    nfilt = 26
    low_freq_mel  = 0
    high_freq_mel = (2595 * np.log10(1 + (sr / 2) / 700))
    mel_points = np.linspace(low_freq_mel, high_freq_mel, nfilt + 2)
    hz_points = 700 * (10**(mel_points / 2595) - 1)
    bin = np.floor((NFFT + 1) * hz_points / sr).astype(int)
    fbank = np.zeros((nfilt, int(np.floor(NFFT / 2 + 1))))
    for m in range(1, nfilt + 1):
        f_m_minus = bin[m - 1]
        f_m       = bin[m]
        f_m_plus  = bin[m + 1]
        for k in range(f_m_minus, f_m):
            fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
        for k in range(f_m, f_m_plus):
            fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])

    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)
    filter_banks = 20 * np.log10(filter_banks)

    mfcc = fftpack.dct(filter_banks, type=2, axis=1, norm='ortho')[:, :13]
    mfcc_mean = np.mean(mfcc, axis=0)

    spectrum = np.abs(np.fft.rfft(segment))
    freqs = np.fft.rfftfreq(len(segment), d=1.0 / sr)

    centroid = np.sum(freqs * spectrum) / np.sum(spectrum)
    bandwidth = np.sqrt(np.sum(((freqs - centroid) ** 2) * spectrum) / np.sum(spectrum))

    cumulative_energy = np.cumsum(spectrum)
    rolloff_threshold = 0.85 * cumulative_energy[-1]
    rolloff_index = np.where(cumulative_energy >= rolloff_threshold)[0][0]
    rolloff = freqs[rolloff_index]

    diff_signal = np.diff(segment)
    jitter = np.mean(np.abs(diff_signal))

    envelope = np.abs(segment)
    shimmer = np.std(np.diff(envelope))

    return mfcc_mean, centroid, bandwidth, rolloff, jitter, shimmer


In [ ]:

# Number of samples per class
num_samples = 200
features = []
labels = []

# Set random seed
np.random.seed(42)

for i in range(num_samples):
    # Real sample generation
    start = np.random.randint(0, len(data) - segment_length)
    segment = data[start:start+segment_length]
    noise = np.random.normal(0, 0.002, segment_length)
    segment_real = segment + noise
    mfcc, centroid, bandwidth, rolloff, jitter, shimmer = compute_features(segment_real, sample_rate)
    features.append(list(mfcc) + [centroid, bandwidth, rolloff, jitter, shimmer])
    labels.append('real')

for i in range(num_samples):
    # Fake sample generation (pitch shifting)
    start = np.random.randint(0, len(data) - segment_length)
    segment = data[start:start+segment_length]
    factor = 0.8 if np.random.rand() < 0.5 else 1.2
    new_length = int(len(segment) / factor)
    resampled = signal.resample(segment, new_length)
    if len(resampled) > segment_length:
        segment_fake = resampled[:segment_length]
    else:
        segment_fake = np.pad(resampled, (0, segment_length - len(resampled)), mode='constant')
    noise = np.random.normal(0, 0.004, segment_length)
    segment_fake = segment_fake + noise
    mfcc, centroid, bandwidth, rolloff, jitter, shimmer = compute_features(segment_fake, sample_rate)
    features.append(list(mfcc) + [centroid, bandwidth, rolloff, jitter, shimmer])
    labels.append('fake')

feature_names = [f'mfcc_{i}' for i in range(13)] + ['spectral_centroid','spectral_bandwidth','spectral_rolloff','jitter','shimmer']
features_df = pd.DataFrame(features, columns=feature_names)
features_df['label'] = labels

# Save to CSV
features_df.to_csv('features.csv', index=False)

# Display the first few rows
display(features_df.head())
